# LoRA as Low-Rank Preconditioning

**Core insight:** LoRA fine-tuning and Nyström kernel approximation both exploit the same mathematical structure — rapid spectral decay in a large matrix.

| | LoRA | Nyström |
|---|---|---|
| **Full object** | Weight update $\Delta W \in \mathbb{R}^{m \times n}$ | Kernel matrix $K \in \mathbb{R}^{N \times N}$ |
| **Low-rank form** | $\Delta W \approx BA$ (rank $r$) | $K \approx C W^\dagger C^\top$ (rank $r$) |
| **Why it works** | Singular values of $\Delta W$ decay rapidly | Eigenvalues of $K$ decay rapidly |
| **Cost reduction** | $\mathcal{O}(dr)$ instead of $\mathcal{O}(d^2)$ | $\mathcal{O}(Nr)$ instead of $\mathcal{O}(N^2)$ |

This notebook demonstrates the connection empirically:
1. Pretrain a base MLP on multi-output regression
2. LoRA fine-tune at ranks $r \in \{2, 4, 8, 16\}$ on a downstream task
3. Full fine-tuning comparison
4. SVD analysis of weight updates — confirming low-rank structure
5. Quantitative Nyström ↔ LoRA correspondence

In [ ]:
%matplotlib inline

import sys, copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

sys.path.insert(0, '.')

from models import BaseModel, LoRAModel, NystromLoRAAnalyzer
from dataset import get_pretrain_loader, get_finetune_loader, FinetuneDataset
from trainer import PretrainTrainer, LoRAFinetuner, FullFinetuner
from nystrom_module import NystromLoRAConnection

np.random.seed(42)
torch.manual_seed(42)

print("Imports OK")

## Phase 1: Pretrain → Fine-tune Pipeline

We pretrain a 3-layer MLP (`64 → 128 → 128 → 8`) on a multi-output regression task built from real sklearn Digits pixel features. The pretrained model learns general representations; we then *adapt* it to a **different** downstream regression target using either LoRA or full fine-tuning.

In [ ]:
torch.manual_seed(42)
base_model = BaseModel(d_in=64, d_hidden=128, d_out=8)
pretrain_loader = get_pretrain_loader(batch_size=32, n_samples=2000)

total_base_params = sum(p.numel() for p in base_model.parameters())
print(f"Base model params: {total_base_params:,}")
print(f"Architecture: 64 → 128 → 128 → 8 (3-layer MLP)")

trainer = PretrainTrainer(base_model, pretrain_loader, lr=1e-3)
pretrain_losses = trainer.train(num_epochs=100)

print(f"Pretrain loss:  {pretrain_losses[0]:.4f} → {pretrain_losses[-1]:.4f}")

pretrained_model = copy.deepcopy(base_model)

finetune_loader = get_finetune_loader(batch_size=16, n_samples=200)
test_set = FinetuneDataset(n_samples=300, seed=999)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=64, shuffle=False)

base_model.eval()
with torch.no_grad():
    baseline_loss = sum(
        nn.MSELoss()(base_model(X), Y).item()
        for X, Y in test_loader
    ) / len(test_loader)
print(f"Pretrained model on finetune test: {baseline_loss:.4f}  ← needs adaptation")

## Phase 2: LoRA Fine-Tuning

**LoRA (Low-Rank Adaptation):** Freeze all base weights $W$ and learn a low-rank update $\Delta W = BA$ where $B \in \mathbb{R}^{m \times r}$, $A \in \mathbb{R}^{r \times n}$, and $r \ll \min(m, n)$.

The effective weight becomes $W_{\text{eff}} = W + BA$. Only $A$ and $B$ are trained — dramatically fewer parameters than full fine-tuning.

In [ ]:
lora_ranks = [2, 4, 8, 16]
lora_results = {}

print(f"{'Rank':>6} {'LoRA Params':>12} {'% of Base':>10} "
      f"{'Final Loss':>12} {'Test Loss':>12}")
print(f"{'-' * 58}")

for rank in lora_ranks:
    torch.manual_seed(42)
    lora_model = LoRAModel(pretrained_model, rank=rank)
    ft_loader = get_finetune_loader(batch_size=16, n_samples=200)
    finetuner = LoRAFinetuner(lora_model, ft_loader, lr=1e-3)
    losses = finetuner.train(num_epochs=40)
    test_metrics = finetuner.evaluate(test_loader)

    n_lora = lora_model.trainable_params()
    pct = 100.0 * n_lora / total_base_params

    lora_results[rank] = {
        "trainable_params": n_lora,
        "pct_of_base": pct,
        "loss_history": losses,
        "test_loss": test_metrics["test_loss"],
    }

    print(f"{rank:>6} {n_lora:>12,} {pct:>9.1f}% "
          f"{losses[-1]:>12.4f} {test_metrics['test_loss']:>12.4f}")

In [ ]:
torch.manual_seed(42)
full_model = copy.deepcopy(pretrained_model)
ft_loader = get_finetune_loader(batch_size=16, n_samples=200)
full_finetuner = FullFinetuner(full_model, ft_loader, lr=1e-3)
full_losses = full_finetuner.train(num_epochs=40)
full_test = full_finetuner.evaluate(test_loader)

full_trainable = sum(p.numel() for p in full_model.parameters())
print(f"Full FT params: {full_trainable:,} (100%)")
print(f"Final loss:     {full_losses[-1]:.4f}")
print(f"Test loss:      {full_test['test_loss']:.4f}")

print(f"\n{'Method':<18} {'Params':>10} {'Train Loss':>12} {'Test Loss':>12}")
print(f"{'-' * 54}")
print(f"{'No finetune':<18} {'0':>10} {'—':>12} {baseline_loss:>12.4f}")
for rank in lora_ranks:
    r = lora_results[rank]
    print(f"{'LoRA r=' + str(rank):<18} {r['trainable_params']:>10,} "
          f"{r['loss_history'][-1]:>12.4f} {r['test_loss']:>12.4f}")
print(f"{'Full FT':<18} {full_trainable:>10,} "
      f"{full_losses[-1]:>12.4f} {full_test['test_loss']:>12.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Pretrain loss
axes[0].plot(pretrain_losses, "k-", lw=2, label="Pretrain")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Phase 1: Pretraining")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel 2: Fine-tune loss curves
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(lora_ranks)))
for i, rank in enumerate(lora_ranks):
    axes[1].plot(lora_results[rank]["loss_history"], "-", color=colors[i],
                 lw=2, label=f"LoRA r={rank}")
axes[1].plot(full_losses, "r--", lw=2, label="Full FT")
axes[1].axhline(baseline_loss, color="gray", ls=":", lw=1, label="No finetune")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MSE Loss")
axes[1].set_title("Phase 2: Fine-Tuning Comparison")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# Panel 3: Parameter efficiency
bar_labels = [f"r={r}" for r in lora_ranks] + ["Full"]
bar_params = [lora_results[r]["trainable_params"] for r in lora_ranks] + [full_trainable]
bar_losses = [lora_results[r]["test_loss"] for r in lora_ranks] + [full_test["test_loss"]]
bar_colors = list(colors) + ["red"]

x_pos = np.arange(len(bar_labels))
bars = axes[2].bar(x_pos, bar_params, color=bar_colors, alpha=0.7, edgecolor="black")
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(bar_labels)
axes[2].set_ylabel("Trainable Parameters")
axes[2].set_title("Parameter Efficiency")
axes[2].grid(True, alpha=0.3, axis="y")

ax2t = axes[2].twinx()
ax2t.plot(x_pos, bar_losses, "k^-", ms=8, lw=2, label="Test Loss")
ax2t.set_ylabel("Test Loss")
ax2t.legend(loc="upper left")

plt.tight_layout()
plt.show()

## Phase 3: SVD Analysis of Weight Updates

**Why does LoRA work?** Because the weight update $\Delta W = W_{\text{finetuned}} - W_{\text{pretrained}}$ is *approximately low-rank*.

We compute the SVD $\Delta W = U \Sigma V^\top$ and examine the singular value spectrum. If $\sigma_1 \gg \sigma_2 \gg \cdots \gg \sigma_r \gg \sigma_{r+1} \approx 0$, then a rank-$r$ approximation $\Delta W \approx U_r \Sigma_r V_r^\top$ captures most of the update — exactly what LoRA does with $BA$.

In [ ]:
svd_stats = NystromLoRAAnalyzer.analyze_rank_structure(
    pretrained_model, full_model
)

print(f"{'Layer':<18} {'Shape':>12} {'‖ΔW‖_F':>10} "
      f"{'r@90%':>8} {'r@95%':>8} {'r@99%':>8} {'σ₁/σₙ':>10}")
print(f"{'-' * 78}")

for name, stats in svd_stats.items():
    er = stats["effective_ranks"]
    shape_str = f"{len(stats['singular_values'])}×{len(stats['singular_values'])}"
    print(f"{name:<18} {shape_str:>12} {stats['full_frobenius_norm']:>10.4f} "
          f"{er['90%']:>8} {er['95%']:>8} {er['99%']:>8} "
          f"{stats['top_sv_ratio']:>10.0f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
layer_names = list(svd_stats.keys())

# Panel 1: Normalized singular value decay
for i, name in enumerate(layer_names):
    sv = np.array(svd_stats[name]["singular_values"])
    sv_norm = sv / (sv[0] + 1e-15)
    axes[0].semilogy(sv_norm, "o-", ms=4, lw=2, label=name.replace(".weight", ""))

axes[0].set_xlabel("Singular Value Index")
axes[0].set_ylabel("σᵢ / σ₁")
axes[0].set_title("Weight Update SVD: Rapid Decay → LoRA is Justified")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Panel 2: Cumulative energy vs rank
for i, name in enumerate(layer_names):
    cumul = np.array(svd_stats[name]["cumulative_energy"])
    er = svd_stats[name]["effective_ranks"]
    axes[1].plot(cumul, "-", lw=2,
                 label=f"{name.replace('.weight', '')} (90% @ r={er['90%']})")

axes[1].axhline(0.90, color="gray", ls="--", alpha=0.5)
axes[1].axhline(0.95, color="gray", ls=":", alpha=0.5)
axes[1].set_xlabel("Rank r")
axes[1].set_ylabel("Cumulative Energy")
axes[1].set_title("Energy vs Rank (justifies low-rank LoRA)")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# Panel 3: Energy captured at specific LoRA ranks
test_ranks_bar = [2, 4, 8, 16]
fc2_sv = svd_stats.get("fc2.weight", list(svd_stats.values())[1])
fc2_cumul = np.array(fc2_sv["cumulative_energy"])
fc2_energies = [fc2_cumul[min(r - 1, len(fc2_cumul) - 1)] * 100
                for r in test_ranks_bar]
fc2_name = "fc2" if "fc2.weight" in svd_stats else list(svd_stats.keys())[1]
bar_c = plt.cm.viridis(np.linspace(0.2, 0.8, len(test_ranks_bar)))
axes[2].bar(range(len(test_ranks_bar)), fc2_energies, color=bar_c,
            edgecolor="black", alpha=0.8)
axes[2].set_xticks(range(len(test_ranks_bar)))
axes[2].set_xticklabels([f"r={r}" for r in test_ranks_bar])
axes[2].set_ylabel("Energy Captured (%)")
axes[2].set_title(f"LoRA Rank vs Energy ({fc2_name})")
axes[2].axhline(90, color="red", ls="--", alpha=0.5, label="90%")
axes[2].axhline(95, color="orange", ls="--", alpha=0.5, label="95%")
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis="y")
axes[2].set_ylim(0, 105)

plt.tight_layout()
plt.show()

## Phase 4: Formal Nyström ↔ LoRA Correspondence

The Nyström method approximates a symmetric PSD matrix $K$ by sampling $r$ columns:
$$K \approx C W^\dagger C^\top$$
where $C = K[:, \mathcal{I}]$ and $W = K[\mathcal{I}, \mathcal{I}]$ for landmark indices $\mathcal{I}$.

We apply this to the **Gram matrix** $G = \Delta W \cdot \Delta W^\top$ (symmetric PSD), whose eigenvalues are $\sigma_i^2$ of $\Delta W$. Both LoRA and Nyström exploit the same rapid spectral decay:

- **LoRA:** learns the best rank-$r$ factors $B, A$ for $\Delta W$
- **Nyström:** samples $r$ columns of $G$ to approximate the full kernel

Below we compare eigendecomposition (optimal) vs. Nyström (column-sampling) errors at each rank.

In [ ]:
nystrom_conn = NystromLoRAConnection()
nystrom_analysis = nystrom_conn.analyze_weight_updates(
    pretrained_model, full_model, ranks=[1, 2, 4, 8, 16]
)

target_layer = "fc2.weight"
if target_layer not in nystrom_analysis:
    target_layer = list(nystrom_analysis.keys())[1]

print(f"Layer: {target_layer}")
print(f"{'Rank':>6} {'SVD Error':>12} {'Energy':>10}")
print(f"{'-' * 30}")
for r, rd in sorted(nystrom_analysis[target_layer]["rank_data"].items()):
    print(f"{r:>6} {rd['svd_relative_error']:>12.4f} "
          f"{rd['energy_captured'] * 100:>9.1f}%")

dW_target = (
    dict(full_model.named_parameters())[target_layer].detach().cpu().numpy()
    - dict(pretrained_model.named_parameters())[target_layer].detach().cpu().numpy()
)

test_ranks = [1, 2, 4, 8, 16, 32]
svd_errors = []
nys_errors_mean = []
nys_errors_std = []

np.random.seed(42)
for r in test_ranks:
    if r > min(dW_target.shape):
        continue
    res = nystrom_conn.nystrom_weight_approximation(dW_target, r, n_trials=20)
    svd_errors.append(res["svd_error"])
    nys_errors_mean.append(res["nystrom_error_mean"])
    nys_errors_std.append(res["nystrom_error_std"])

valid_ranks = test_ranks[:len(svd_errors)]

print(f"\nNyström vs eigendecomp of Gram matrix G = ΔW·ΔWᵀ ({target_layer}):")
print(f"{'Rank':>6} {'Eig Error':>12} {'Nyström Mean':>14} {'Nyström Std':>14}")
print(f"{'-' * 48}")
for i, r in enumerate(valid_ranks):
    print(f"{r:>6} {svd_errors[i]:>12.4f} {nys_errors_mean[i]:>14.4f} "
          f"{nys_errors_std[i]:>14.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Eigendecomp vs Nyström error
axes[0].semilogy(valid_ranks, svd_errors, "b^-", ms=8, lw=2,
                 label="Eigendecomp (optimal)")
axes[0].errorbar(valid_ranks, nys_errors_mean, yerr=nys_errors_std,
                 fmt="rs-", ms=8, lw=2, capsize=4,
                 label="Nyström (column sampling)")
axes[0].set_xlabel("Rank r")
axes[0].set_ylabel("Relative Frobenius Error")
axes[0].set_title(f"Gram Matrix G = ΔW·ΔWᵀ ({target_layer.replace('.weight', '')})")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Panel 2: Singular value decay of ΔW
sv = np.array(nystrom_analysis[target_layer]["singular_values"])
axes[1].semilogy(sv / sv[0], "k-o", ms=3, lw=2)
for r in [2, 4, 8]:
    if r < len(sv):
        axes[1].axvline(r, color="gray", ls="--", alpha=0.4)
        axes[1].annotate(f"r={r}", (r, sv[r] / sv[0]), fontsize=8)
axes[1].set_xlabel("Index i")
axes[1].set_ylabel("σᵢ / σ₁")
axes[1].set_title("Singular Value Decay of ΔW")
axes[1].grid(True, alpha=0.3)

# Panel 3: Formal correspondence table
analogy = [
    ["Full matrix", "A ∈ R^{N×N} (PDE op.)", "ΔW ∈ R^{d×d} (wt. update)"],
    ["Low-rank", "CW†C^T (Nyström)", "BA (LoRA)"],
    ["Why works", "Eigenvalue decay", "Singular value decay"],
    ["Cost: full", "O(N²)", "O(d²)"],
    ["Cost: low-rank", "O(Nr)", "O(dr)"],
    ["Landmarks", "Column samples", "Learned rank dirs"],
]
axes[2].axis("off")
table = axes[2].table(
    cellText=analogy,
    colLabels=["Concept", "Nyström (PDE)", "LoRA (Fine-tuning)"],
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.0, 1.6)
for key, cell in table.get_celld().items():
    if key[0] == 0:
        cell.set_facecolor("#4472C4")
        cell.set_text_props(color="white", fontweight="bold")
    else:
        cell.set_facecolor("#D6E4F0" if key[0] % 2 == 1 else "#EDF2F9")
axes[2].set_title("Formal Nyström ↔ LoRA Correspondence", pad=20)

plt.tight_layout()
plt.show()

## Summary

| Concept | Nyström (PDE / Kernel) | LoRA (Fine-tuning) |
|---|---|---|
| **Full matrix** | $A \in \mathbb{R}^{N \times N}$ (PDE operator) | $\Delta W \in \mathbb{R}^{d \times d}$ (weight update) |
| **Low-rank form** | $C W^\dagger C^\top$ (Nyström, rank $r$) | $BA$ (LoRA, rank $r$) |
| **Why it works** | Eigenvalue decay of $A$ | Singular value decay of $\Delta W$ |
| **Cost: full** | $\mathcal{O}(N^2)$ | $\mathcal{O}(d^2)$ |
| **Cost: low-rank** | $\mathcal{O}(Nr)$ | $\mathcal{O}(dr)$ |
| **Landmarks** | Column samples from kernel matrix | Learned rank directions ($B$, $A$) |

**Key takeaway:** Both LoRA and Nyström are instances of the same principle — when a matrix has rapidly decaying spectrum, a rank-$r$ approximation captures most of the information at a fraction of the cost. LoRA *learns* the optimal low-rank factors while Nyström *samples* them, but the mathematical justification is identical: spectral concentration.